# 05 · Base policy: Diffusion Policy trained on the scripted demos

The steering work keeps a strong generative policy *frozen* and steers it at inference time, so
we first need a policy. This is a **Diffusion Policy** (Chi et al. 2023): a 1D-conv U-Net over
action chunks with FiLM conditioning, DDPM training and DDIM sampling (`plugsim/learn/policy.py`).

Observation modes:
* `state`: proprioception (44) + privileged task features (plug pose, target-port pose, and
  relative errors). This stands in for perception, trains in minutes on a laptop, and is the
  **local default**.
* `image`: proprioception + gripper-camera frames (small CNN with spatial-softmax keypoints). This
  is the realistic setting, meant for GPU nodes (more demos, long training).

What it took to make BC work on this precision task (each was a 0 % → working change):
1. **Predict the clean action chunk** (x0 / "sample" prediction) instead of the noise. With a
   laptop-scale budget, ε-prediction stayed at cm-level error.
2. **Relative targets**: each chunk is a sequence of controller targets relative to the current
   target, with robust (percentile) scaling, so fine near-goal motions are small, precisely learnt
   numbers. Absolute world targets or env deltas both failed.
3. **DART noise** (Laskey et al. 2017) in the demos: execute expert action + noise and record the
   expert's clean corrective action, so the data shows how to recover from small errors.

Expect a policy that is clearly *worse* than the privileged expert. That's useful, because it
leaves room for steering (notebook 07).

In [3]:
# Setup cell, inlined into every notebook by notebooks/build.py.
# Locally: finds the repo. On a Colab runtime (also VS Code connected to Colab), where the code is not
# on the VM: fetches it from (1) PLUGSIM_REPO (git URL, default: the public repo), (2) /content/RobotPluggingSim.zip, or
# (3) MyDrive/RobotPluggingSim (Drive), installs it, and keeps data/ + runs/ on Drive when mounted.
import os, sys, pathlib, subprocess
DEFAULT_REPO = "https://github.com/carlo-scr/RobotPluggingSim.git"
PLUGSIM_REPO = os.environ.get("PLUGSIM_REPO") or DEFAULT_REPO  # Colab: code is cloned from here
USE_DRIVE = os.environ.get("PLUGSIM_DRIVE", "1") == "1"  # set False if drive.mount hangs in your client
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    # numpy's BLAS: one thread (the sim solves tiny systems; extra threads only add contention)
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    # Colab ships the NVIDIA driver but not its EGL vendor file, so EGL silently falls back to CPU
    # (software) rendering, ~100x slower. Register the NVIDIA EGL library (as in MuJoCo's Colab tutorial).
    _icd = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
    if subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0 and not _icd.exists():
        _icd.parent.mkdir(parents=True, exist_ok=True)
        _icd.write_text('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
        print("registered the NVIDIA EGL vendor file (GPU rendering)")
    CODE = pathlib.Path("/content/RobotPluggingSim")
    DRIVE = pathlib.Path("/content/drive/MyDrive")
    if (CODE / ".git").exists():
        subprocess.run(["git", "-C", str(CODE), "pull", "-q"])  # pick up pushed changes
    if not (CODE / "plugsim").exists():
        if PLUGSIM_REPO:
            subprocess.run(["git", "clone", "-q", PLUGSIM_REPO, str(CODE)], check=True)
        elif pathlib.Path("/content/RobotPluggingSim.zip").exists():
            subprocess.run(["unzip", "-q", "-o", "/content/RobotPluggingSim.zip", "-d", "/content"], check=True)
    if USE_DRIVE and not DRIVE.exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:  # e.g. not supported by the client; fine with PLUGSIM_REPO or the zip
            print("Drive not mounted:", e)
    if not (CODE / "plugsim").exists():
        if (DRIVE / "RobotPluggingSim").exists():
            subprocess.run(["rsync", "-a", "--exclude", ".venv", "--exclude", "data", "--exclude", "runs", "--exclude", "media",
                            f"{DRIVE}/RobotPluggingSim/", f"{CODE}/"], check=True)
        else:  # last resort: the public repo
            subprocess.run(["git", "clone", "-q", DEFAULT_REPO, str(CODE)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)
    if DRIVE.exists():  # persist datasets and checkpoints across sessions / notebooks
        for d in ("data", "runs"):
            tgt = DRIVE / "plugsim_runs" / d
            tgt.mkdir(parents=True, exist_ok=True)
            link = CODE / d
            if not link.is_symlink():
                subprocess.run(["rm", "-rf", str(link)])
                link.symlink_to(tgt)
    else:
        print("WARNING: no Drive: data/ and runs/ live on the VM and are lost when the runtime stops")
    try:
        import torch
        if torch.cuda.is_available():
            os.environ.setdefault("PLUGSIM_SCALE", "a100")
    except ImportError:
        pass
    os.chdir(CODE)
ROOT = pathlib.Path.cwd()
while not (ROOT / "plugsim").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "egl")  # headless GPU nodes
print("ROOT:", ROOT, "| colab:", IN_COLAB, "| scale:", os.environ.get("PLUGSIM_SCALE", "local"))

Mounted at /content/drive
ROOT: /content/RobotPluggingSim | colab: True | scale: a100


In [4]:
import os, sys, pathlib, subprocess, time, json
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from plugsim.env import PlugEnv
from plugsim.scenario import sample_scenario
from plugsim.expert import ScriptedExpert
from plugsim.learn.data import list_episodes, load_many
from plugsim.learn.policy import PolicyCfg, train_policy, save_policy, load_policy, PolicyRunner, rollout
from plugsim.tools import video_html, run_live

SCALE = os.environ.get("PLUGSIM_SCALE", "local")
NCPU = max(1, (os.cpu_count() or 2) - 1)
OBS = os.environ.get("PLUGSIM_OBS", "state")  # "state" | "image"
P = {
    "local": dict(n_demos=1000, workers=min(7, NCPU), dart=0.0025, steps=10000, bs=256, n_eval=20, obs_mode=OBS,
                  device="mps" if torch.backends.mps.is_available() else "cpu"),
    "a100": dict(n_demos=4000, workers=NCPU, dart=0.0025, steps=60000, bs=1024, n_eval=50, obs_mode=OBS, device="cuda"),  # Colab A100
    "h100": dict(n_demos=10000, workers=NCPU, dart=0.0025, steps=200000, bs=1024, n_eval=200, obs_mode=OBS, device="cuda"),
}[SCALE]
IMG_CAMS = ("right_gripper_front", "left_gripper_front", "head_front")
DATA, RUNS = ROOT / "data" / f"demos_{P['obs_mode']}_{SCALE}", ROOT / "runs"
CKPT = RUNS / f"policy_{P['obs_mode']}.pt"
RUNS.mkdir(exist_ok=True)
print(SCALE, P)

a100 {'n_demos': 4000, 'workers': 11, 'dart': 0.0025, 'steps': 60000, 'bs': 1024, 'n_eval': 50, 'obs_mode': 'state', 'device': 'cuda'}


## 1 · Demonstrations (nominal scripted expert + DART noise)
Re-running after a disconnect is cheap: cached demos and a saved checkpoint are reused. If the
kernel restarted, run **all** cells from the top again (the first cells only re-import).

In [5]:
print("cached demos:", (DATA / "index_0.parquet").exists(), "| cached policy:", CKPT.exists(), "->", CKPT)
if not (DATA / "index_0.parquet").exists() and not (CKPT.exists() and not os.environ.get("RETRAIN")):
    cams = "none" if P["obs_mode"] == "state" else ",".join(IMG_CAMS)
    cfg_over = json.dumps({"camera.width": 128, "camera.height": 72})
    t = time.time()
    # streams progress; resumes if interrupted (re-run this cell); aborts with a message if workers get stuck
    run_live([sys.executable, "-u", str(ROOT / "scripts/generate_dataset.py"), "--out", str(DATA), "--n", str(P["n_demos"]), "--workers", str(P["workers"]),
              "--mix", "nominal:1", "--cameras", cams, "--dart", str(P["dart"]), "--no-states", "--cfg", cfg_over])
    print(f"({time.time() - t:.0f}s)")
NEED_TRAIN = not CKPT.exists() or bool(os.environ.get("RETRAIN"))
if NEED_TRAIN:  # demos are only needed for training
    idx = list_episodes(DATA)
    idx = idx[idx.outcome == "success"]
    eps = load_many(idx.path.tolist(), cams=IMG_CAMS if P["obs_mode"] == "image" else (), frames=P["obs_mode"] == "image", progress=False)
    print(f"{len(eps)} successful demos, {sum(len(e['action']) for e in eps)} steps")

cached demos: True | cached policy: True -> /content/RobotPluggingSim/runs/policy_state.pt


## 2 · Train (or load a cached checkpoint)

In [6]:
cfg = PolicyCfg(obs_mode=P["obs_mode"], cams=IMG_CAMS, action_mode="rel", prediction="sample", ddim_eta=0.0)
if not NEED_TRAIN:
    pol = load_policy(CKPT)
    print("loaded", CKPT)
else:
    t = time.time()
    pol, hist = train_policy(eps, cfg, steps=P["steps"], bs=P["bs"], lr=1e-3, device=P["device"], log_every=max(1, P["steps"] // 10))
    save_policy(pol, CKPT)
    print(f"trained in {time.time() - t:.0f}s -> {CKPT}")
    plt.figure(figsize=(6, 3)); plt.semilogy(pd.Series(hist).rolling(100).mean()); plt.xlabel("step"); plt.ylabel("x0 MSE (normalised)"); plt.show()
pol = pol.cpu().eval()
print(f"{sum(p.numel() for p in pol.parameters()) / 1e6:.2f} M parameters")

loaded /content/RobotPluggingSim/runs/policy_state.pt
5.13 M parameters


## 3 · Closed-loop evaluation on held-out scenarios (vs. the privileged expert)

In [7]:
env = PlugEnv(render=False)
runner = PolicyRunner(pol, env, seed=0)
rows, t0 = [], time.time()
for s in range(P["n_eval"]):
    sc = sample_scenario(env.cfg, 900_000 + s)
    r = rollout(env, runner, sc)
    pr = env.privileged()
    env.reset(sc); ex = ScriptedExpert(env, seed=900_000 + s)
    while not ex.done and env.time < 16: env.step(ex.act())
    rows.append(dict(seed=900_000 + s, policy=r["success"], expert=env.latched_port == sc.port_idx, policy_latch_s=r["latch_time"], port=sc.port_idx))
R = pd.DataFrame(rows)
print(f"policy success {R.policy.mean():.0%}  |  expert success {R.expert.mean():.0%}  ({time.time() - t0:.0f}s)")
R.head(10)

policy success 76%  |  expert success 98%  (1034s)


,seed,policy,expert,policy_latch_s,port
0,900000,True,True,4.55,17
1,900001,True,True,5.50,35
2,900002,True,True,5.10,13
3,900003,True,True,5.95,10
4,900004,True,True,5.55,31
5,900005,True,True,5.25,6
6,900006,True,True,5.20,41
7,900007,False,True,NaN,38
8,900008,True,True,5.25,30
9,900009,True,True,5.00,8


## 4 · Policy rollouts: a success and a failure

In [ ]:
# The policy samples stochastically, so a re-run of an evaluation seed is a new draw: record rollouts
# until we have one success and one failure (at ~3/4 success that takes a few tries).
envr = PlugEnv()
def record(seed, runner_seed):
    runner_r = PolicyRunner(pol, envr, seed=runner_seed)
    sc = sample_scenario(envr.cfg, seed)
    envr.reset(sc)
    frames, t_latch = [], None
    while envr.time < 16:
        envr.step(runner_r.act())
        if envr.step_count % 2 == 0:
            arm = "right" if envr.privileged()["grasp_pos"][1] < 0 else "left"
            frames.append(np.concatenate([envr.render_free(lookat=(0.4, 0, 0.95), distance=1.5, azimuth=90, elevation=-30, width=384, height=216),
                                          envr.render(f"{arm}_gripper_front", 384, 216)], 1))
        if envr.latched_port >= 0 and t_latch is None: t_latch = envr.time
        if t_latch and envr.time - t_latch > 1: break
    return envr.latched_port == sc.port_idx, frames

clips = {}
for k, seed in enumerate(R.seed.tolist()[:12]):
    ok, frames = record(int(seed), runner_seed=k)
    print(f"seed {seed}: {'success' if ok else 'failure'}")
    clips.setdefault(ok, frames)
    if len(clips) == 2: break

In [ ]:
video_html(clips.get(True, next(iter(clips.values()))), fps=10, width=760)  # a success

In [ ]:
video_html(clips[False], fps=10, width=760) if False in clips else print("no failure in these tries")  # a failure